In [1]:
import numpy as np 
import matplotlib.pyplot as plt 
import pandas as pd 
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

In [2]:
data = pd.read_csv('titanic/train.csv')
data_test = pd.read_csv('titanic/test.csv')
data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Dealing with missing values and removing unsufficient features

- Missing values in "Embarked" column are replaced with mode, as only a small number of values are missing.

- Additionally, "Ticket" and "Cabin" columns are removed, since they contatin a large amount of missing values and difficult to transform into meaningful patterns.

In [3]:
data.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [4]:
data['Embarked'] = data['Embarked'].fillna(data['Embarked'].mode()[0])
data = data.drop(['Ticket', 'Cabin'], axis = 1)

data_test['Embarked'] = data_test['Embarked'].fillna(data_test['Embarked'].mode()[0])
data_test = data_test.drop(['Ticket', 'Cabin'], axis = 1)

data.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Fare             0
Embarked         0
dtype: int64

## Feature Engineering

A few preprocessing and feature engineering steps were applied to improve the predictivness of dataset.

### Categorical features

- The "Sex" feature was transformed into numerical one using binary encoding.

- The "Embarked" feature was one-hot-encoded

### Title Extraction

Passenger titles were extracted from "Name" feature. Rare titles were grouped into one "No Title" category. Then a new "Title" column were also one-hot-encoded.

### Other features

2 new features were created: "SizeOfFamily" and "IsAlone". 
- The first one is the total number of sibling/spouses and parent/children traveling together. 
- The second one is the a binary indicator if the passanger traveled alone.

Additionally, missing values in "Age" column were filled with meadian age among each Title group, allowing more precise estimination.

Finally, the "Name" column was removed as it should not include any useful information.

In [5]:
data['Sex'] = data['Sex'].map({'male': 0, 'female': 1})
data_test['Sex'] = data_test['Sex'].map({'male': 0 , 'female': 1})

data = pd.get_dummies(data, columns=['Embarked'])
data_test = pd.get_dummies(data_test, columns=['Embarked'])

data['SizeOfFamily'] = data['SibSp'] + data['Parch']
data_test['SizeOfFamily'] = data_test['SibSp'] + data_test['Parch']

data['IsAlone'] = (data['SizeOfFamily'] == 0).astype(int)
data_test['IsAlone'] = (data_test['SizeOfFamily'] == 0).astype(int)

common_titles = ['Mr', 'Mrs', 'Miss', 'Dr', 'Master']
data['Title'] = data['Name'].apply(lambda x: x.split(',')[1].split('.')[0].strip())  # Surname, Title. Name -> [Surname, Rest] -> [Title, Name] -> Title + remove any leading whitespaces
data['Title'] = data['Title'].apply(lambda x: x if x in common_titles else 'No Title')
data_test['Title'] = data_test['Name'].apply(lambda x: x.split(',')[1].split('.')[0].strip())
data_test['Title'] = data_test['Title'].apply(lambda x: x if x in common_titles else 'No Title')

data['Age'] = data.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))
data_test['Age'] = data_test.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))

data = pd.get_dummies(data, columns=['Title'])
data_test = pd.get_dummies(data_test, columns=['Title'])

data = data.drop('Name', axis=1)
data_test = data_test.drop('Name', axis=1)

## Model Training and Parameter Optimization

The dataset was dividet into: X - input deatures, and y - target variable.

Several machine learning algorithms, such as: Logistic Regression, Random Forest, XGBoost, Naive Bayes, KNN, SVM with polynomial and rbf kernels, were evaluated to compare their performance on our dataset.

For each model hyperparameter tunning was performed using 'GridSearchCV' using 10-fold cross-validation.

Accuracy was used as the primary evaluation metric, as our task is basically binary classification.

In [6]:
X = data.drop('Survived', axis = 1)
y = data['Survived']
X

,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S,SizeOfFamily,IsAlone,Title_Dr,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_No Title
0,1,3,0,22.0,1,0,7.2500,False,False,True,1,0,False,False,False,True,False,False
1,2,1,1,38.0,1,0,71.2833,True,False,False,1,0,False,False,False,False,True,False
2,3,3,1,26.0,0,0,7.9250,False,False,True,0,1,False,False,True,False,False,False
3,4,1,1,35.0,1,0,53.1000,False,False,True,1,0,False,False,False,False,True,False
4,5,3,0,35.0,0,0,8.0500,False,False,True,0,1,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,2,0,27.0,0,0,13.0000,False,False,True,0,1,False,False,False,False,False,True
887,888,1,1,19.0,0,0,30.0000,False,False,True,0,1,False,False,True,False,False,False
888,889,3,1,21.0,1,2,23.4500,False,False,True,3,0,False,False,True,False,False,False
889,890,1,0,26.0,0,0,30.0000,True,False,False,0,1,False,False,False,True,False,False


In [7]:
LogReg = LogisticRegression(max_iter=10000, random_state=22)
parameter_grid_logreg = {'solver': ['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'], 'C': [0.1, 1, 10]}
grid_search_log = GridSearchCV(LogReg, param_grid=parameter_grid_logreg, cv=10, scoring='accuracy')
grid_search_log.fit(X, y)
print(f'Logistic Regression best score using 10fold cross validations is: {grid_search_log.best_score_} using parameters {grid_search_log.best_params_}')

RF = RandomForestClassifier(random_state=22)
parameter_grid_rf = {'n_estimators': [25, 50, 100], 'max_depth': [None, 10, 15, 25], 'min_samples_split': [10, 15, 25], 'max_features': ['sqrt', 'log2', None], 'min_samples_leaf': [1,2,5]}
grid_search_rf = GridSearchCV(RF, param_grid=parameter_grid_rf, cv=10, scoring='accuracy', n_jobs = -1)
grid_search_rf.fit(X, y)
print(f'Random Forest best score using 10fold cross validations is: {grid_search_rf.best_score_} using parameters {grid_search_rf.best_params_}')

xgb = XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state = 22)
parameter_grid_xgb = {'n_estimators': [50, 100, 200], 'max_depth': [None,15, 25, 35], 'learning_rate': [0.01, 0.1, 0.001], 'subsample': [0.8, 1.0], 'colsample_bytree': [0.8, 1.0]}
grid_search_xgb = GridSearchCV(xgb, param_grid=parameter_grid_xgb, cv=10, scoring='accuracy', n_jobs = -1)
grid_search_xgb.fit(X, y)
print(f'XGBoost best score using 10fold cross validations is: {grid_search_xgb.best_score_} using parameters {grid_search_xgb.best_params_}')

naive_byes = GaussianNB()
parameter_grid_nb = {'var_smoothing': [1e-9, 1e-7, 1e-5]}
grid_search_nb = GridSearchCV(naive_byes, param_grid=parameter_grid_nb, cv=10, scoring='accuracy')
grid_search_nb.fit(X, y)
print(f'NaiveByes best score using 10fold cross validations is: {grid_search_nb.best_score_} using parameters {grid_search_nb.best_params_}')

scaler=StandardScaler()
X_scaled = scaler.fit_transform(X)

knn = KNeighborsClassifier()
parameter_grid_knn = {'n_neighbors': [25, 30, 50, 75, 100], 'weights': ['uniform', 'distance'], 'p': [1,2]}
grid_search_knn = GridSearchCV(knn, param_grid=parameter_grid_knn, cv=10, scoring='accuracy', n_jobs = -1)
grid_search_knn.fit(X_scaled, y)
print(f'KNN best score using 10fold cross validations is: {grid_search_knn.best_score_} using parameters {grid_search_knn.best_params_}')


Logistic Regression best score using 10fold cross validations is: 0.8327715355805243 using parameters {'C': 10, 'solver': 'liblinear'}
Random Forest best score using 10fold cross validations is: 0.8395380774032459 using parameters {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'min_samples_split': 10, 'n_estimators': 100}
XGBoost best score using 10fold cross validations is: 0.8417977528089887 using parameters {'colsample_bytree': 1.0, 'learning_rate': 0.01, 'max_depth': None, 'n_estimators': 200, 'subsample': 0.8}
NaiveByes best score using 10fold cross validations is: 0.8069787765293382 using parameters {'var_smoothing': 1e-07}
KNN best score using 10fold cross validations is: 0.8271535580524343 using parameters {'n_neighbors': 50, 'p': 2, 'weights': 'distance'}


In [8]:
svmp = SVC(random_state=22, kernel= 'poly')
parameter_grid_svmp = {'C': [100000], 'degree': [2, 3]} #
grid_search_svmp = GridSearchCV(svmp, param_grid=parameter_grid_svmp, cv=10, scoring='accuracy', n_jobs = -1)
grid_search_svmp.fit(X, y)
print(f'SVM best score using 10fold cross validations is: {grid_search_svmp.best_score_} using parameters {grid_search_svmp.best_params_}')

SVM best score using 10fold cross validations is: 0.7891885143570537 using parameters {'C': 100000, 'degree': 2}


In [9]:
svmrbf = SVC(random_state=22, kernel= 'rbf')
parameter_grid_svmrbf = {'C': [100000], 'gamma' : ['scale', 'auto']} 
grid_search_svmrbf = GridSearchCV(svmrbf, param_grid=parameter_grid_svmrbf, cv=10, scoring='accuracy', n_jobs = -1)
grid_search_svmrbf.fit(X, y)
print(f'SVM best score using 10fold cross validations is: {grid_search_svmrbf.best_score_} using parameters {grid_search_svmrbf.best_params_}')

SVM best score using 10fold cross validations is: 0.8070037453183521 using parameters {'C': 100000, 'gamma': 'scale'}


## Feature Importance Analysis

To better understand the behaviour of 2 best models, feature analysis was performed

In [10]:
best_rf = grid_search_rf.best_estimator_
importances_rf = best_rf.feature_importances_
features = X.columns

feature_importance_rf = pd.Series(importances_rf, index=features).sort_values(ascending=False)
print(feature_importance_rf)

Sex               0.197992
Title_Mr          0.196540
Fare              0.114909
Pclass            0.100729
PassengerId       0.069798
Age               0.069094
Title_Miss        0.058702
SizeOfFamily      0.048426
Title_Mrs         0.043594
SibSp             0.029580
Parch             0.015218
Embarked_S        0.012194
Title_Master      0.010451
Embarked_C        0.010228
IsAlone           0.010192
Embarked_Q        0.007044
Title_No Title    0.005309
Title_Dr          0.000000
dtype: float64


In [11]:
best_xgb = grid_search_xgb.best_estimator_
importances_xgb = best_xgb.feature_importances_
features = X.columns

feature_importance_xgb = pd.Series(importances_xgb, index=features).sort_values(ascending=False)
print(feature_importance_xgb)

Title_Mr          0.490563
Sex               0.157667
Pclass            0.067932
Title_Master      0.051877
SizeOfFamily      0.045544
Title_No Title    0.034569
SibSp             0.021460
Fare              0.020665
Embarked_S        0.017383
Age               0.014598
Embarked_Q        0.012746
Title_Dr          0.012633
PassengerId       0.012449
Title_Miss        0.012371
Embarked_C        0.010044
Title_Mrs         0.009075
Parch             0.008424
IsAlone           0.000000
dtype: float32


## Final Prediction

After selecting the best-performing model, prediciton were generated for the kaggle dataset.

In [12]:
data_test['Fare'] = data_test['Fare'].fillna(data_test['Fare'].mode()[0])
predictions = grid_search_xgb.best_estimator_.predict(data_test)

submission = pd.DataFrame({'PassengerId': data_test['PassengerId'], 'Survived': predictions})

submission.to_csv("submission.csv", index=False)

## Final Results

The final submission achieved:

- Testset accutacy: **0.78** accuracy
- Kaggle placement: **2818, top 22%**